In [2]:
import cv2


def mosaic_face(img):
    # load the face detector
    faceCascade = cv2.CascadeClassifier(cv2.data.haarcascades 
                                    + "haarcascade_frontalface_default.xml")

    # create another variable for the result image
    result = img.copy()

    # detect faces
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    faces = faceCascade.detectMultiScale(gray, 1.3, 5)

    for (x, y, w, h) in faces:
        face = img[y:y + h, x:x + w, :]  # extract face region

        s = 10  # down-sampling interval / tiny block size
        downsampled_face = face[::s, ::s, :]  # down-sampling

        # nearest upsampling to the original face size
        mosaic_face = cv2.resize(
            downsampled_face,
            (w, h),
            interpolation=cv2.INTER_NEAREST)

        # modify face region
        result[y:y + h, x:x + w, :] = mosaic_face

    return result


if __name__ == "__main__":
    img = cv2.imread("tst1.jpg")
    result = mosaic_face(img)
    cv2.imshow('original', img)
    cv2.imshow('mosaic', result)
    cv2.waitKey(0)

In [3]:
import cv2
import numpy as np


def sharpening(img):
    # sharpening kernel
    kernel_sharpen_1 = np.array([[-1, -1, -1],
                                 [-1, 9, -1],
                                 [-1, -1, -1]])
    output = cv2.filter2D(img, -1, kernel_sharpen_1)
    return output


def motion_blur(img):
    # motion blur kernel size
    size = 11
    # generating the kernel
    kernel_motion_blur = np.zeros((size, size))
    kernel_motion_blur[int((size - 1) / 2), :] = np.ones(size)
    kernel_motion_blur = kernel_motion_blur / size

    # applying the kernel to the input image
    output = cv2.filter2D(img, -1, kernel_motion_blur)
    return output


if __name__ == "__main__":
    img = cv2.imread("tst2.jpg")
    result1 = sharpening(img)
    result2 = motion_blur(img)

    cv2.imshow('original', img)
    cv2.imshow('sharpening', result1)
    cv2.imshow('motion blur', result2)
    cv2.waitKey(0)


In [4]:
import cv2


def cartoonize_image(img):
    # Convert image to grayscale
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Apply median filter to the grayscale image
    img_gray = cv2.medianBlur(img_gray, 7)

    # Detect edges in the image and threshold the result
    edges = cv2.Laplacian(img_gray, cv2.CV_8U, ksize=5)
    ret, mask = cv2.threshold(edges, 80, 255, cv2.THRESH_BINARY_INV)
    cv2.imshow('mask', mask)

    # From the documentation of bilateralFilter:
    # ----
    #   For simplicity, you can set the 2 sigma values to be the same. 
    #   If they are small (< 10), the filter will not have much effect, 
    #   whereas if they are large (> 150), they will have a very strong 
    #   effect, making the image look "cartoonish".
    # ----
    sigma_color = 220
    sigma_space = 220
    size = 15

    # Apply bilateral filter
    filtered_img = cv2.bilateralFilter(img, size, sigma_color, sigma_space)
    cv2.imshow('filtered_img', filtered_img)
    # Add the thick boundary lines to the image using 'AND' operator
    dst = cv2.bitwise_and(filtered_img, filtered_img, mask=mask)

    return dst


if __name__ == "__main__":
    img = cv2.imread("tst3.jpg")
    result = cartoonize_image(img)
    cv2.imshow('original', img)
    cv2.imshow('cartoonized', result)
    cv2.waitKey(0)
